In [1]:
import os
import torch
import json
from PIL import Image
import torchvision
import torch.nn.functional as F
import torchvision.transforms as T
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models import mobilenet_v3_large, mobilenet_v3_small
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform


In [2]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from torchvision.ops import box_iou
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, average_precision_score

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
train_image_folder = "/content/drive/MyDrive/ThesisDataset/Splits/train"
val_image_folder = "/content/drive/MyDrive/ThesisDataset/Splits/val"
test_image_folder = "/content/drive/MyDrive/ThesisDataset/Splits/test"

train_annotation_file = "/content/drive/MyDrive/ThesisDataset/Splits/train.json"
val_annotation_file = "/content/drive/MyDrive/ThesisDataset/Splits/val.json"
test_annotation_file = "/content/drive/MyDrive/ThesisDataset/Splits/test.json"

In [41]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained = True)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [42]:
mobilenet_v3 = mobilenet_v3_large(pretrained=True)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [43]:
backbone = mobilenet_v3.features

In [44]:
class MobileNetV3Backbone(nn.Module):
    def __init__(self, backbone, out_channels=256):
        super(MobileNetV3Backbone, self).__init__()
        self.backbone = backbone
        self.out_channels = out_channels
        self.conv = nn.Conv2d(960, self.out_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = self.backbone(x)
        x = self.conv(x)
        return {"0": x}

custom_backbone = MobileNetV3Backbone(backbone=mobilenet_v3.features)

In [45]:
model = FasterRCNN(
    custom_backbone,
    num_classes=8,
)

In [46]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

model.transform = GeneralizedRCNNTransform(
    min_size=[224],
    max_size=224,
    image_mean=mean,
    image_std=std
)

In [47]:
# anchor_sizes = ((16, 32), (64, 96), (128, 192))
# aspect_ratios = ((0.5, 1.0, 2.0),)

# anchor_sizes = ((16, 32), (64, 96), (128, 192))
# aspect_ratios = ((0.33, 0.5, 1.0, 2.0, 3.0),)

anchor_sizes = ((32,), (64,),)
aspect_ratios = ((0.5, 1.0, 2.0),)

anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)
model.rpn.anchor_generator = anchor_generator

In [37]:
model

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=[224], max_size=224, mode='bilinear')
  )
  (backbone): MobileNetV3Backbone(
    (backbone): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(16, eps

In [13]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, json_file, image_dir, category_map=None):
        with open(json_file) as f:
            self.data = json.load(f)
        self.image_dir = image_dir

        self.indices = [img['id'] for img in self.data['images']]
        self.images = {img['id']: img['file_name'] for img in self.data['images']}

        self.annotations = {ann['image_id']: [] for ann in self.data['annotations']}

        if category_map is None:
            category_map = {
                1: 1,
                2: 2,
                5: 5,
                12: 3,
                14: 4,
                16: 6,
                19: 7
            }

        self.category_map = category_map

        for ann in self.data['annotations']:
            x, y, w, h = ann['bbox']
            x2, y2 = x + w, y + h
            mapped_category_id = self.category_map.get(ann['category_id'], ann['category_id'])
            self.annotations[ann['image_id']].append({
                'bbox': [x, y, x2, y2],
                'category_id': mapped_category_id
            })

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, indx):
        image_id = self.indices[indx]
        image_path = f"{self.image_dir}/{self.images[image_id]}"

        img = Image.open(image_path)
        mean = [0.5]
        std = [0.5]

        transform = T.Compose([
            T.Grayscale(num_output_channels=3),
            T.ToTensor(),
            T.Normalize(mean=mean * 3, std=std * 3)
            ])

        img = transform(img)

        annotations = self.annotations[image_id]

        boxes = [ann['bbox'] for ann in annotations]
        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)

        labels = torch.tensor([ann['category_id'] for ann in annotations], dtype=torch.int64)

        target = {'boxes': boxes_tensor, 'labels': labels}

        return img, target

def coco_collate_fn(batch):
    return batch

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [15]:
train_loader = torch.utils.data.DataLoader(
    CustomDataset(json_file= train_annotation_file, image_dir= train_image_folder),
    batch_size=32,
    shuffle=True,
    collate_fn=coco_collate_fn,
    pin_memory= True if device.type == 'cuda' else False
)

In [16]:
for batch in train_loader:
    images, targets = zip(*batch)
    print(f"Number of images in batch: {len(images)}")
    print(f"First image shape: {images[0].shape}")
    print(f"First target: {targets[0]}")
    break


Number of images in batch: 32
First image shape: torch.Size([3, 224, 224])
First target: {'boxes': tensor([[113.4802,  72.0352, 169.7269, 125.3216],
        [ 47.3656,   0.0000,  94.7313,  37.4978]]), 'labels': tensor([5, 5])}


In [17]:
val_loader = DataLoader(
    CustomDataset(json_file= val_annotation_file, image_dir= val_image_folder),
    batch_size=8,
    shuffle=True,
    collate_fn=coco_collate_fn,
    pin_memory=True if device.type == 'cuda' else False
)

In [18]:
for batch in val_loader:
    images, targets = zip(*batch)
    print(f"Number of images in batch: {len(images)}")
    print(f"First image shape: {images[0].shape}")
    print(f"First target: {targets[0]}")
    break

Number of images in batch: 8
First image shape: torch.Size([3, 224, 224])
First target: {'boxes': tensor([[136.5000, 110.6000, 206.5000, 138.2500],
        [121.4500,  67.9000, 172.2000,  95.9000]]), 'labels': tensor([7, 7])}


In [48]:
epoches = 15

In [49]:
model = model.to(device)

In [21]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0005
)

In [22]:
scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

In [40]:
model_save_folder = '/content/drive/MyDrive/ThesisTraining/Model-Approach23'

if not os.path.exists(model_save_folder):
    os.makedirs(model_save_folder)

In [50]:
for epoch in range(epoches):
    model.train()
    running_loss = 0.0
    print(f"Epoch {epoch + 1}/{epoches}")
    progress_bar = tqdm(train_loader, desc=f"Training", ncols=80)

    for batch in progress_bar:
        images, targets = zip(*batch)
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += losses.item()
        progress_bar.set_postfix({
            "loss": f"{losses.item():.4f}"
        })

    avg_train_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epoches} - Training Loss: {avg_train_loss:.4f}")

    scheduler.step()

    model.eval()
    with torch.no_grad():
        running_iou_loss = 0.0
        total_batches = 0

        for batch in val_loader:
            try:
                images, targets = zip(*batch)
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                outputs = model(images)
                iou_loss = 0.0
                for o, t in zip(outputs, targets):
                    if o["boxes"].size(0) == 0 or t["boxes"].size(0) == 0:
                        continue
                    min_size = min(o["boxes"].size(0), t["boxes"].size(0))
                    iou_loss += F.smooth_l1_loss(o["boxes"][:min_size], t["boxes"][:min_size])

                if len(outputs) > 0:
                    iou_loss /= len(outputs)
                    running_iou_loss += iou_loss.item()
                    total_batches += 1

            except RuntimeError as e:
                print(f"Error during validation on batch: {e}")
                continue

        if total_batches > 0:
            avg_iou_loss = running_iou_loss / total_batches
        else:
            avg_iou_loss = 0.0
        print(f"Epoch {epoch + 1}/{epoches} - Validation IoU Loss: {avg_iou_loss:.4f}")

    model_save_path = os.path.join(model_save_folder, f'model_epoch_{epoch + 1}.pth')

    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_train_loss,
    }, model_save_path)

print(f"Training Completed")

Epoch 1/15


Training: 100%|████████████████████| 56/56 [28:03<00:00, 30.06s/it, loss=2.9355]


Epoch 1/15 - Training Loss: 2.9101
Epoch 1/15 - Validation IoU Loss: 49.8487
Epoch 2/15


Training: 100%|████████████████████| 56/56 [18:25<00:00, 19.74s/it, loss=2.8944]


Epoch 2/15 - Training Loss: 2.9098
Epoch 2/15 - Validation IoU Loss: 49.8540
Epoch 3/15


Training: 100%|████████████████████| 56/56 [18:22<00:00, 19.69s/it, loss=2.8636]


... (11 outputs left out)

Training: 100%|████████████████████| 56/56 [18:24<00:00, 19.73s/it, loss=2.8634]


Epoch 9/15 - Training Loss: 2.9096
Epoch 9/15 - Validation IoU Loss: 52.1499
Epoch 10/15


Training:   7%|█▌                   | 4/56 [01:29<19:27, 22.45s/it, loss=2.9076]


KeyboardInterrupt: 